# 06 — Gold: FactOrderFulfillment (Accumulating Snapshot) — Spark SQL

Grain: uma linha por `OrderID`.

**Técnica Spark SQL:** `MERGE INTO` com `WHEN MATCHED AND condition THEN UPDATE SET ...`.

Quando `ShippedDate` chega, a linha é atualizada — sem DeltaTable API.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 06 FactOrderFulfillment")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:51:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


In [3]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW src_fulfillment AS
    SELECT
        ABS(HASH(o.OrderID))                                               AS FulfillmentSK,
        o.OrderID,
        dc.CustomerSK,
        de.EmployeeSK,
        ds.ShipperSK,
        CAST(DATE_FORMAT(CAST(o.OrderDate    AS DATE), 'yyyyMMdd') AS INT) AS OrderDateKey,
        CAST(DATE_FORMAT(CAST(o.RequiredDate AS DATE), 'yyyyMMdd') AS INT) AS RequiredDateKey,
        CASE WHEN o.ShippedDate IS NOT NULL
             THEN CAST(DATE_FORMAT(CAST(o.ShippedDate AS DATE), 'yyyyMMdd') AS INT)
        END                                                                AS ShippedDateKey,
        o.Freight,
        o.ShipCountry,
        CASE WHEN o.ShippedDate IS NOT NULL
             THEN DATEDIFF(CAST(o.ShippedDate AS DATE), CAST(o.OrderDate AS DATE))
        END                                                                AS DaysToShip,
        CASE WHEN o.ShippedDate IS NOT NULL
             THEN CAST(o.ShippedDate AS DATE) > CAST(o.RequiredDate AS DATE)
        END                                                                AS IsLate,
        current_timestamp()                                                AS LoadTimestamp
    FROM bronze.orders o
    JOIN gold.DimCustomer dc
      ON  o.CustomerID = dc.CustomerID
      AND CAST(o.OrderDate AS DATE) >= dc.ValidFrom
      AND CAST(o.OrderDate AS DATE) <  dc.ValidTo
    JOIN gold.DimEmployee de ON o.EmployeeID = de.EmployeeID
    LEFT JOIN gold.DimShipper ds ON o.ShipVia = ds.ShipperID
""")

n       = spark.sql("SELECT COUNT(*) AS n FROM src_fulfillment").collect()[0]["n"]
shipped = spark.sql("SELECT COUNT(*) AS n FROM src_fulfillment WHERE ShippedDateKey IS NOT NULL").collect()[0]["n"]
print(f"Source FactOrderFulfillment: {n} linhas (esperado: 830)")
print(f"  Enviados: {shipped} | Pendentes: {n - shipped}")

Source FactOrderFulfillment: 830 linhas (esperado: 830)
  Enviados: 809 | Pendentes: 21


In [4]:
spark.sql("""
    MERGE INTO gold.FactOrderFulfillment AS tgt
    USING src_fulfillment AS src
    ON tgt.OrderID = src.OrderID
    WHEN MATCHED AND (
        (tgt.ShippedDateKey IS NULL     AND src.ShippedDateKey IS NOT NULL) OR
        (tgt.ShippedDateKey IS NOT NULL AND tgt.ShippedDateKey <> src.ShippedDateKey)
    )
    THEN UPDATE SET
        tgt.ShippedDateKey = src.ShippedDateKey,
        tgt.DaysToShip     = src.DaysToShip,
        tgt.IsLate         = src.IsLate,
        tgt.LoadTimestamp  = src.LoadTimestamp
    WHEN NOT MATCHED THEN INSERT *
""")

n = spark.sql("SELECT COUNT(*) AS n FROM gold.FactOrderFulfillment").collect()[0]["n"]
print(f"FactOrderFulfillment após MERGE: {n} linhas (esperado: 830)")

FactOrderFulfillment após MERGE: 830 linhas (esperado: 830)


In [5]:
# ============================================================
# LAB-A: Encontrar pedido pendente e mostrar estado antes
# Padrão Accumulating Snapshot: a linha EXISTE mas ShippedDateKey=NULL
# ============================================================
row = spark.sql("""
    SELECT OrderID, OrderDateKey, RequiredDateKey
    FROM gold.FactOrderFulfillment
    WHERE ShippedDateKey IS NULL
    LIMIT 1
""").collect()

if row:
    oid = row[0].OrderID
    print(f"OrderID={oid} — sem ShippedDate. Estado atual (ANTES):")
    spark.sql(f"""
        SELECT OrderID, OrderDateKey, RequiredDateKey,
               ShippedDateKey, DaysToShip, IsLate
        FROM gold.FactOrderFulfillment
        WHERE OrderID = {oid}
    """).show()
else:
    print("Todos os pedidos já foram enviados. Nada a simular.")
    oid = None


OrderID=11008 — sem ShippedDate. Estado atual (ANTES):


+-------+------------+---------------+--------------+----------+------+
|OrderID|OrderDateKey|RequiredDateKey|ShippedDateKey|DaysToShip|IsLate|
+-------+------------+---------------+--------------+----------+------+
|  11008|    19980408|       19980506|          NULL|      NULL|  NULL|
+-------+------------+---------------+--------------+----------+------+



In [6]:
# ============================================================
# LAB-B: Atualizar bronze + re-executar MERGE inline
# ============================================================
if oid:
    # Simular chegada de ShippedDate no bronze
    spark.sql(f"""
        UPDATE bronze.orders
        SET ShippedDate = CAST('2026-03-15' AS TIMESTAMP)
        WHERE OrderID = {oid}
    """)
    print(f"bronze.orders atualizado: OrderID={oid} → ShippedDate=2026-03-15")

    # Re-executar a VIEW fonte (invalida cache Spark SQL para pegar bronze atualizado)
    spark.catalog.refreshTable("bronze.orders")
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW src_fulfillment AS
        SELECT
            ABS(HASH(o.OrderID))                                               AS FulfillmentSK,
            o.OrderID,
            dc.CustomerSK,
            de.EmployeeSK,
            ds.ShipperSK,
            CAST(DATE_FORMAT(CAST(o.OrderDate    AS DATE), 'yyyyMMdd') AS INT) AS OrderDateKey,
            CAST(DATE_FORMAT(CAST(o.RequiredDate AS DATE), 'yyyyMMdd') AS INT) AS RequiredDateKey,
            CASE WHEN o.ShippedDate IS NOT NULL
                 THEN CAST(DATE_FORMAT(CAST(o.ShippedDate AS DATE), 'yyyyMMdd') AS INT)
            END                                                                AS ShippedDateKey,
            o.Freight, o.ShipCountry,
            CASE WHEN o.ShippedDate IS NOT NULL
                 THEN DATEDIFF(CAST(o.ShippedDate AS DATE), CAST(o.OrderDate AS DATE))
            END                                                                AS DaysToShip,
            CASE WHEN o.ShippedDate IS NOT NULL
                 THEN CAST(o.ShippedDate AS DATE) > CAST(o.RequiredDate AS DATE)
            END                                                                AS IsLate,
            current_timestamp()                                                AS LoadTimestamp
        FROM bronze.orders o
        JOIN gold.DimCustomer dc
          ON  o.CustomerID = dc.CustomerID
          AND CAST(o.OrderDate AS DATE) >= dc.ValidFrom
          AND CAST(o.OrderDate AS DATE) <  dc.ValidTo
        JOIN gold.DimEmployee de ON o.EmployeeID = de.EmployeeID
        LEFT JOIN gold.DimShipper ds ON o.ShipVia = ds.ShipperID
    """)

    # Re-executar MERGE
    spark.sql("""
        MERGE INTO gold.FactOrderFulfillment AS tgt
        USING src_fulfillment AS src
        ON tgt.OrderID = src.OrderID
        WHEN MATCHED AND (
            (tgt.ShippedDateKey IS NULL     AND src.ShippedDateKey IS NOT NULL) OR
            (tgt.ShippedDateKey IS NOT NULL AND tgt.ShippedDateKey <> src.ShippedDateKey)
        )
        THEN UPDATE SET
            tgt.ShippedDateKey = src.ShippedDateKey,
            tgt.DaysToShip     = src.DaysToShip,
            tgt.IsLate         = src.IsLate,
            tgt.LoadTimestamp  = src.LoadTimestamp
        WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"Estado DEPOIS do MERGE:")
    spark.sql(f"""
        SELECT OrderID, OrderDateKey, RequiredDateKey,
               ShippedDateKey, DaysToShip, IsLate
        FROM gold.FactOrderFulfillment
        WHERE OrderID = {oid}
    """).show()


bronze.orders atualizado: OrderID=11008 → ShippedDate=2026-03-15


Estado DEPOIS do MERGE:


+-------+------------+---------------+--------------+----------+------+
|OrderID|OrderDateKey|RequiredDateKey|ShippedDateKey|DaysToShip|IsLate|
+-------+------------+---------------+--------------+----------+------+
|  11008|    19980408|       19980506|      20260315|     10203|  true|
+-------+------------+---------------+--------------+----------+------+



In [7]:
# ============================================================
# LAB-C: Reset para manter idempotência do notebook
# ============================================================
if oid:
    spark.sql(f"""
        UPDATE bronze.orders
        SET ShippedDate = NULL
        WHERE OrderID = {oid}
    """)
    spark.sql(f"""
        UPDATE gold.FactOrderFulfillment
        SET ShippedDateKey = NULL,
            DaysToShip     = NULL,
            IsLate         = NULL
        WHERE OrderID = {oid}
    """)
    print(f"Reset concluído (OrderID={oid}). Notebook idempotente.")


Reset concluído (OrderID=11008). Notebook idempotente.


In [8]:
print("Validações:")
n = spark.sql("SELECT COUNT(*) AS n FROM gold.FactOrderFulfillment").collect()[0]["n"]
print(f"  Total: {n} (esperado: 830)")

dups = spark.sql("""
    SELECT COUNT(*) AS n FROM (
        SELECT OrderID FROM gold.FactOrderFulfillment GROUP BY OrderID HAVING COUNT(*) > 1
    )
""").collect()[0]["n"]
print(f"  Grain único: {dups} duplicatas (esperado: 0)")

spark.sql("""
    SELECT (ShippedDateKey IS NULL) AS IsPending, COUNT(*) AS count
    FROM gold.FactOrderFulfillment GROUP BY IsPending
""").show()

spark.sql("""
    SELECT ROUND(AVG(DaysToShip), 1) AS avg_days,
           MIN(DaysToShip) AS min_days, MAX(DaysToShip) AS max_days
    FROM gold.FactOrderFulfillment WHERE DaysToShip IS NOT NULL
""").show()

Validações:


  Total: 830 (esperado: 830)


  Grain único: 0 duplicatas (esperado: 0)


+---------+-----+
|IsPending|count|
+---------+-----+
|    false|  809|
|     true|   21|
+---------+-----+



+--------+--------+--------+
|avg_days|min_days|max_days|
+--------+--------+--------+
|     8.5|       1|      37|
+--------+--------+--------+

